In [6]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage, BaseMessage
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
class JokeState(TypedDict):
    topic: str
    joke: Annotated[list[str], add_messages]
    explaination: str

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"

llm = HuggingFaceEndpoint(
    repo_id=model_id,
    task="text-generation",
    max_new_tokens=256,
    huggingfacehub_api_token="",
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1,
)

model = ChatHuggingFace(llm=llm)

d:\Learner\Recap\Lang Graph\01_Foundation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
def gen_joke(state: JokeState) ->JokeState:
    prompt = [HumanMessage(f"make a joke on {state['topic']}")]

    result = model.invoke(prompt)

    return {
        'joke': [result]
    }

def explain_joke(state: JokeState) ->JokeState:
    prompt = [HumanMessage(f"explain the joke")]

    result = model.invoke(prompt)

    return {
        'explaination': result
    }

In [16]:
checkpointer = InMemorySaver()

graph = StateGraph(JokeState)

graph.add_node('make_joke', gen_joke)
graph.add_node('explain_joke', explain_joke)

graph.add_edge(START, 'make_joke')
graph.add_edge('make_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
thread_id = '1'

config = {
    'configurable': {'thread_id': thread_id}
}

workflow.invoke({'topic': 'Equality and UpperCast and Reservation in India'}, config=config)

{'topic': 'Equality and UpperCast and Reservation in India',
 'joke': [AIMessage(content="A sensitive topic! Here's a joke that tries to tackle the complexities of equality, upper cast, and reservation in India:\n\n**Why did the Dalit's reservation form go to therapy?**\n\nBecause it was struggling to balance its **equality** expectations with the **upper** cast's sense of entitlement! (get it?)\n\nBut seriously, the joke highlights the complexities of the Indian caste system and the ongoing struggle for equality and social justice.\n\n**Why did the Reservation Committee go to a party?**\n\nBecause they wanted to **level** the playing field and show that **everyone** deserves a chance, regardless of their **caste** or **social status**!\n\n**Why did the Dalit's equality advocate break up with his girlfriend?**\n\nBecause she was always trying to **upper** him, and he wanted to **level** the relationship!\n\n(Sorry, I know these jokes are a bit of a stretch, but I hope they bring a smil